# Chapter 2: Agentic Foundations

Agents are not just LLM calls wrapped in a loop. They are **purposeful entities**
with declared capabilities, memory hierarchies, and self-correction mechanisms.

**Concepts covered:**
- PEAS Framework (Performance, Environment, Actuators, Sensors)
- Hot / Warm / Cold Memory Hierarchy
- Reflexion (iterative self-correction)
- Token budget management

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

from core.agents.base import AgentCapability, BaseAgent, CognitiveDomain, ResourceConstraints
from core.agents.reflexion import ReflexionAgent, ReflexionResult
from core.memory.hierarchy import ColdMemory, HotMemory, MemoryEntry, WarmMemory
print('Agentic foundations package ready.')

## 2.1 PEAS Framework

Every well-designed agent declares **four things** at construction time:
- **P**erformance: what outcome does success look like?
- **E**nvironment: what knowledge sources and context can it perceive?
- **A**ctuators: what capabilities (tools) is it allowed to use?
- **S**ensors: what events can it observe?

> If an agent cannot answer "what am I allowed to do?", it will eventually do something unexpected.

In [ ]:
# Define the cognitive domain (Environment in PEAS)
order_domain = CognitiveDomain(
    name='OrderManagement',
    description='Processes customer orders, validates inventory, emits order events',
    knowledge_sources=['orders-db', 'inventory-api'],
    tools=['get_order', 'update_order_status', 'check_inventory'],
)

# Define resource constraints
constraints = ResourceConstraints(
    max_tokens_per_call=2_048,
    max_calls_per_minute=30,
    token_budget_per_session=50_000,
)

# Create the agent with declared capabilities
agent = BaseAgent(
    cognitive_domain=order_domain,
    capabilities=frozenset({AgentCapability.READ_DATA, AgentCapability.EMIT_EVENTS}),
    constraints=constraints,
)

print(f'Agent ID: {agent.agent_id[:8]}...')
print(f'Domain: {agent.cognitive_domain.name}')
print(f'Capabilities: {[c.name for c in agent.capabilities]}')

# Test scope validation
import asyncio
in_scope = asyncio.run(agent.validate_scope('get_order for customer 123'))
out_scope = asyncio.run(agent.validate_scope('delete all users from database'))
print(f'\nIn-scope task: {in_scope}')
print(f'Out-of-scope task: {out_scope}')

## 2.2 The Memory Hierarchy

Agents need memory at three temperatures:

| Tier | Speed | Capacity | Purpose |
|------|-------|----------|---------|
| **Hot** | Instant | Small (~4K tokens) | Active context window |
| **Warm** | Fast | Medium (thousands) | RAG-style dynamic retrieval |
| **Cold** | Slower | Unlimited | Long-term knowledge store |

In Azure: Warm maps to **Azure AI Search**, Cold maps to **Azure Cosmos DB**.

In [ ]:
# Hot Memory: token-budgeted FIFO context window
hot = HotMemory(max_tokens=100, avg_chars_per_token=4)

# Simulate a conversation accruing context
conversation = [
    'User: What is the status of order ORD-123?',
    'Agent: Order ORD-123 is currently being processed.',
    'User: When will it ship?',
    'Agent: Expected ship date is tomorrow.',
    'User: Can I change the delivery address?',
    'Agent: Yes, provide the new address.',
]
for turn in conversation:
    hot.add(MemoryEntry(text=turn))

context = hot.get_context()
print(f'Token budget: 100 tokens (~400 chars)')
print(f'Context entries retained: {len(context)}')
print(f'Context (truncated):')
for t in context[-3:]:
    print(f'  {t}')

In [ ]:
# Warm Memory: semantic retrieval with stub embeddings
# In production, replace with Azure OpenAI text-embedding-3-small
warm = WarmMemory(top_k=3)

# Index domain knowledge
knowledge_chunks = [
    ('circuit_breaker_intro', 'Circuit breakers prevent cascading failures in microservices'),
    ('event_sourcing', 'Event sourcing stores state as a sequence of immutable events'),
    ('saga_pattern', 'Sagas coordinate distributed transactions without 2PC'),
    ('bounded_context', 'Bounded contexts define explicit linguistic and semantic boundaries'),
    ('cqrs_pattern', 'CQRS separates read and write models for independent scaling'),
]

for key, text in knowledge_chunks:
    warm.index(MemoryEntry(text=text, metadata={'key': key}))

# Semantic retrieval
query = 'distributed transaction coordination'
results = warm.retrieve(query, top_k=2)
print(f'Query: "{query}"')
print(f'Top {len(results)} results:')
for r in results:
    print(f'  - {r.text}')

In [ ]:
# Cold Memory: persistent key-value store with tag search
cold = ColdMemory()

# Store agent configuration and domain policies
cold.write('policy:order-agent', MemoryEntry(
    text='Order agent must validate inventory before accepting orders.',
    metadata={'tags': ['policy', 'orders'], 'version': 2}
))
cold.write('policy:payment-agent', MemoryEntry(
    text='Payment agent must use 3DS2 for transactions above EUR 30.',
    metadata={'tags': ['policy', 'payments'], 'version': 1}
))
cold.write('knowledge:microservices-principles', MemoryEntry(
    text='Services should be independently deployable, loosely coupled, and organized around business capabilities.',
    metadata={'tags': ['architecture', 'principles']}
))

# Tag-based search
policy_entries = cold.search_by_tag('policy')
print(f'Policies stored: {len(policy_entries)}')
for e in policy_entries:
    print(f'  - {e.text[:60]}...')

# Direct key lookup
order_policy = cold.read('policy:order-agent')
print(f'\nOrder policy version: {order_policy.metadata["version"]}')

## 2.3 Reflexion: Iterative Self-Correction

Single-shot LLM calls produce mediocre results. **Reflexion** introduces a feedback loop:

1. **Generate**: produce initial response
2. **Score**: evaluate quality (0.0–1.0)
3. **Reflect**: identify what was wrong
4. **Regenerate**: improve with reflection as guidance

This mirrors how a senior engineer reviews and iterates on a junior's PR.

In [ ]:
import asyncio

# Simulated LLM functions (no real API call needed for the demo)
attempt_scores = [0.3, 0.6, 0.9]  # Progressive improvement
call_count = [0]

async def mock_generate(prompt: str) -> str:
    call_count[0] += 1
    quality = ['vague answer', 'better answer with context', 'precise answer with examples']
    return quality[min(call_count[0] - 1, 2)]

async def mock_reflect(response: str, task: str) -> str:
    return f'The response needs more specifics and a concrete example for: {task}'

score_iter = [0]
async def mock_score(response: str) -> float:
    s = attempt_scores[min(score_iter[0], 2)]
    score_iter[0] += 1
    return s

agent = ReflexionAgent(
    generate_fn=mock_generate,
    reflect_fn=mock_reflect,
    score_fn=mock_score,
    quality_threshold=0.85,
    max_iterations=3,
)

async def main():
    result = await agent.run('Explain the circuit breaker pattern')
    print(f'Converged: {result.converged}')
    print(f'Iterations: {result.iterations}')
    print(f'Final score: {result.quality_score:.2f}')
    print(f'\nIteration trace:')
    for step in result.trace:
        print(f'  [{step["iteration"]}] score={step["score"]:.2f} → {step["response"]}')

asyncio.run(main())

## 2.4 Azure Integration

To use real Azure OpenAI in the Reflexion loop:

```python
import os
from openai import AsyncAzureOpenAI

if os.getenv('AZURE_OPENAI_ENDPOINT'):
    client = AsyncAzureOpenAI(
        azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
        api_key=os.environ['AZURE_OPENAI_API_KEY'],
        api_version='2024-10-21',
    )
    deployment = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')

    async def azure_generate(prompt: str) -> str:
        resp = await client.chat.completions.create(
            model=deployment,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=512,
        )
        return resp.choices[0].message.content

    # Replace mock_generate with azure_generate in the agent above
```

Warm memory in production uses **Azure AI Search** vectorized on `text-embedding-3-small`.